# Day 16 Revision Summary — Decision Trees & Random Forests

- A **decision tree** is a flowchart of yes/no questions on one feature at a time (e.g. `worst_radius < 17?`) that the model learns itself — its big advantage is **interpretability**: you can literally read the reasoning.
- Trees pick splits using **Gini impurity** (`Gini = 1 − Σ pᵢ²`): 0 = pure group (all one class), 0.5 = the most mixed a 2-class group can be. At each node the tree greedily searches every feature/threshold and keeps the split with the lowest weighted child impurity (highest **information gain**).
- `tree.feature_importances_` shows what the model actually used (e.g. "worst radius" did 76% of the work on breast_cancer) — a checkable, human-readable insight a black-box model can't give you.
- A **random forest** grows many trees on bootstrap-sampled rows and random feature subsets, then lets them **vote** (classification) or average (regression) — bagging. Test score went from 0.939 (single depth-3 tree) to 0.956 (200-tree forest), even though the forest still perfectly fits training data (1.000).
- **The bake-off**: on `breast_cancer`, the simple Week-3 logistic regression baseline (`0.981 ± 0.007`) beat both the tree (`0.919 ± 0.025`) and the forest (`0.958 ± 0.024`) — always race a fancy model against a simple one, and weigh accuracy against speed, interpretability, and reliability, not just the top score. Trees/forests also need **no feature scaling**, since they split on order, not distance.

*Resource note: `formulas.md` at the Day 16 folder root has the full first-principles derivations (Gini, information gain, the CART split-search algorithm, feature importance, the forest vote, bootstrap/OOB sampling, and the bias-variance argument for why averaging trees reduces error) — referenced here, not reproduced in full.*

## Classwork Exercise 1 — Grow a Tree (`decision_tree.py`, ~15 min)

**What's being asked:** Fit a `DecisionTreeClassifier(max_depth=3)` on `breast_cancer`, print train and test scores, then try `max_depth=None` and watch it overfit (a Day-14 callback). Print the top-5 features from `feature_importances_`. Verified reference: `max_depth=3` → train 0.976 / test 0.939. Stretch: use `export_text` to print the actual learned flowchart.

**Approach:**
1. Load `load_breast_cancer(as_frame=True)`; split with `train_test_split(..., test_size=0.2, random_state=42, stratify=y)`.
2. Fit `DecisionTreeClassifier(max_depth=3, random_state=42)`. No scaling needed for trees.
3. Print `tree.score(X_train, y_train)` and `tree.score(X_test, y_test)`.
4. Re-fit with `max_depth=None` and print the same two scores — confirm training hits ~1.000 while test drops (overfitting).
5. Sort `tree.feature_importances_` (paired with feature names) and print the top 5.
6. Stretch: `from sklearn.tree import export_text; print(export_text(tree, feature_names=list(X.columns)))`.

In [ ]:
"""
Day 16 · Classwork Exercise 1 -- Grow a tree
Reference: max_depth=3 -> train ~0.976, test ~0.939
Top feature (max_depth=3): worst radius ~0.764 importance
"""
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

# TODO 1: load as a DataFrame, build X, y, and split (test_size=0.2, random_state=42, stratify=y)
data = load_breast_cancer(as_frame=True)
X, y = None, None  # data.data, data.target

# TODO 2: fit DecisionTreeClassifier(max_depth=3, random_state=42) -- no scaling needed
tree = None

# TODO 3: print tree.score(X_train, y_train) and tree.score(X_test, y_test)

# TODO 4: re-fit with max_depth=None; print train/test scores again -- see the overfit
tree_full = None

# TODO 5: sort tree.feature_importances_ (paired with X.columns) and print the top 5

# TODO 6 (stretch): print(export_text(tree, feature_names=list(X.columns)))


## Classwork Exercise 2 — Forest + Bake-off (`random_forest.py` + `bakeoff.py`, ~20 min)

**What's being asked:** Fit a `RandomForestClassifier(n_estimators=200)`, print train/test scores and top-5 importances. Then build a bake-off: cross-validate the tree, the forest, and the Week-3 logistic-regression pipeline, printing each as `mean ± std`, and decide which one you'd actually ship. Verified reference: forest train 1.000 / test 0.956; bake-off — tree `0.919 ± 0.025`, forest `0.958 ± 0.024`, logistic `0.981 ± 0.007` (logistic wins).

**Approach:**
1. Fit `RandomForestClassifier(n_estimators=200, random_state=42)` on the same split as Exercise 1; print train/test scores and top-5 `feature_importances_`.
2. Build three candidates: `DecisionTreeClassifier(max_depth=3, random_state=42)`, the random forest above, and a `Pipeline(StandardScaler, LogisticRegression)` (from Week 3).
3. Run `cross_val_score(model, X, y, cv=5)` for each of the three on the full `X, y` (not just the train split).
4. Print each as `mean ± std` and identify the winner.
5. Write one sentence: which model would you ship for this problem, and why — is the highest score always the right choice (consider speed, interpretability, reliability)?

In [ ]:
"""
Day 16 · Classwork Exercise 2 -- Random forest + the bake-off
Reference: forest train ~1.000, test ~0.956
Bake-off (5-fold CV): tree ~0.919+/-0.025, forest ~0.958+/-0.024, logistic ~0.981+/-0.007 (wins)
"""
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target

# TODO 1: split (test_size=0.2, random_state=42, stratify=y), fit
# RandomForestClassifier(n_estimators=200, random_state=42); print train/test scores
# and the top-5 feature_importances_
forest = None

# TODO 2: build the three bake-off candidates
tree_candidate = None       # DecisionTreeClassifier(max_depth=3, random_state=42)
forest_candidate = None     # RandomForestClassifier(n_estimators=200, random_state=42)
logistic_pipe = None        # Pipeline([("scale", StandardScaler()),
                             #           ("clf", LogisticRegression(max_iter=1000))])

# TODO 3: cross_val_score(candidate, X, y, cv=5) for each of the three

# TODO 4: print each as "mean +/- std" and note which one wins

# TODO 5: write a one-sentence comment on which model you'd actually ship, and why


## Homework Exercise 1 — Bake-off on Wine (~15 min)

**What's being asked:** Run the same bake-off (tree vs forest vs logistic pipeline) on `load_wine` instead of `breast_cancer`. Does the winner change?

**Approach:**
1. Load `load_wine(return_X_y=True)`.
2. Reuse the same three model candidates from Classwork Exercise 2 (tree, forest, logistic pipeline) — remember `LogisticRegression` may need `max_iter` increased and/or `multi_class` handling for 3 classes, which sklearn handles automatically.
3. Run `cross_val_score(candidate, X, y, cv=5)` for each on the wine data.
4. Print each as `mean ± std` and compare the ranking to the breast-cancer bake-off — did the winner change?

In [ ]:
"""
Day 16 · Homework 1 -- Bake-off on load_wine
"""
from sklearn.datasets import load_wine
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# TODO 1: load X, y from load_wine(return_X_y=True)
X, y = None, None

# TODO 2: build the same three candidates as Classwork Exercise 2
tree_candidate = None
forest_candidate = None
logistic_pipe = None

# TODO 3: cross_val_score(candidate, X, y, cv=5) for each

# TODO 4: print each as "mean +/- std"; compare the ranking to breast_cancer --
# did the winner change on this dataset?


## Homework Exercise 2 — Importances at Depth 3 vs Unlimited (~15 min)

**What's being asked:** Print (or plot) `feature_importances_` for a tree at `max_depth=3` versus `max_depth=None` and describe how the ranking/weights shift.

**Approach:**
1. Fit two `DecisionTreeClassifier`s on the same `breast_cancer` split from Classwork Exercise 1: one with `max_depth=3`, one with `max_depth=None`.
2. Extract `feature_importances_` from each, paired with feature names.
3. Print (or plot with a simple bar chart) the top 5-10 features for both, side by side.
4. Write a short observation: does the deeper tree spread importance across more features, or does one feature still dominate? Why might that be, given how greedy splitting works?

In [ ]:
"""
Day 16 · Homework 2 -- Feature importances at max_depth=3 vs None
"""
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target

# TODO 1: split (test_size=0.2, random_state=42, stratify=y)

# TODO 2: fit tree_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
# and tree_full = DecisionTreeClassifier(max_depth=None, random_state=42) on X_train, y_train
tree_depth3 = None
tree_full = None

# TODO 3: print the top 5-10 feature_importances_ for both trees, side by side
# (optional: a simple matplotlib bar chart comparing the two)

# TODO 4: write a comment -- how does the importance ranking/spread shift between
# the shallow and the unlimited-depth tree? Why might that happen?


## Other homework items (no code needed)

- **Read `formulas.md`** in today's folder — Gini impurity, information gain, the CART split-search algorithm, feature importance, the random-forest vote, bootstrap/out-of-bag sampling, and why averaging trees reduces variance — first principles.
- **Commit your work:** `git add . && git commit -m "day 16"`.